# Spark Lab 4: Understanding Spark Execution

**Data Analysis with PySpark**


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("Spark_Lab4_Execution_Study").getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Spark version: 4.2.0
Spark UI: [local Spark UI URL omitted]


## Part 1 - Lazy Evaluation (10 Points)


In [ ]:
df = spark.range(1, 1000001)
df2 = df.filter("id % 2 = 0")
df3 = df2.withColumn("squared", df2.id * df2.id)

### Answers

1. **Has Spark processed all one million records after creating `df3`?** No. Creating `df`, `df2`, and `df3` defines a source and a chain of transformations, but does not evaluate every row.
2. **Why?** DataFrame transformations are lazy. Spark records the operations in a plan and waits for an action that needs the result.
3. **Are `filter()` and `withColumn()` transformations or actions?** Both are transformations. They return new DataFrames and do not by themselves return the computed rows to the driver.


In [3]:
df3.count()

500000

[Screenshot output omitted from public copy; see README.]

### Continue the Answers

4. **What changed when `count()` was called?** Spark optimized and executed the plan needed to count the even IDs. The result should be `500000`.
5. **Why is `count()` an action?** It requires Spark to run the plan and return a scalar result to the driver.
6. **How can lazy evaluation help optimize an application?** Spark can combine compatible operations, push filters or column selection closer to the data source when supported, and avoid work that does not contribute to the requested result.


## Part 2 - Partitions and Parallel Processing (15 Points)


In [4]:
original_partition_count = df.rdd.getNumPartitions()
print("Original df partitions:", original_partition_count)

df4 = spark.range(1, 1000001, numPartitions=4)
print("df4 partitions:", df4.rdd.getNumPartitions())

partition_sizes = df4.rdd.glom().map(len).collect()
print("Records in each df4 partition:", partition_sizes)
print("Total records across partitions:", sum(partition_sizes))

Original df partitions: 16
df4 partitions: 4
Records in each df4 partition: [250000, 250000, 250000, 250000]
Total records across partitions: 1000000


[Screenshot output omitted from public copy; see README.]

### Answers

1. **How many partitions did the original DataFrame contain?** The original DataFrame contained **16 partitions**.
2. **How many partitions does `df4` contain?** `df4` contained **4 partitions**.
3. **What is a Spark partition?** A partition is a logical chunk of a distributed dataset that Spark can process independently as part of a stage.
4. **Why divide large datasets into partitions?** Partitions let Spark schedule work in parallel across available cores and executors, and let it recover or retry work at a smaller unit than the entire dataset.
5. **Partition -> Task -> Executor:** A task processes one partition for one stage. The scheduler assigns tasks to executor processes, which run them using their available cores.
6. **If a stage contains 20 partitions, approximately how many tasks will Spark create?** About 20 tasks for that stage, normally one task per partition. Retries or speculative attempts can make the number of task attempts larger.


The observed partition sizes were **[250000, 250000, 250000, 250000]**.


## Part 3 - Narrow vs. Wide Transformations (15 Points)


In [ ]:
base = spark.range(1, 1000001, numPartitions=4)
filtered = base.filter(col("id") > 500000)

grouped = (
    base
    .withColumn("group", col("id") % 10)
    .groupBy("group")
    .count()
)

### Answers

1. **Is `filter()` narrow or wide?** Usually narrow. Each output row depends on one input row from its existing partition, so it does not need to bring matching rows together from other partitions.
2. **Is `groupBy()` narrow or wide?** Wide. Rows with the same grouping key may start in different partitions and must be brought together for the final aggregation.
3. **Which operation is more likely to move data between partitions?** `groupBy()`.
4. **What is this movement called?** A shuffle.
5. **Why are shuffles expensive?** They can require serialization, network transfer, sorting or aggregation, disk writes, and disk reads. The cost depends on data size, partitioning, and cluster resources.

**Complete the flow:**

- Narrow transformation -> data stays within its input partition -> usually no shuffle.
- Wide transformation -> data is redistributed between partitions -> shuffle.

**Examples:** Narrow: `filter`, `select`, `map`. Wide: `groupBy`, `distinct`, `repartition`, and many joins. A particular operation's plan can depend on its expression and input partitioning.


## Part 4 - Spark Execution Plan (15 Points)


In [6]:
grouped.explain()
grouped.explain(True)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[group#16L], functions=[count(1)])
   +- Exchange hashpartitioning(group#16L, 200), ENSURE_REQUIREMENTS
      +- HashAggregate(keys=[group#16L], functions=[partial_count(1)])
         +- Project [(id#12L % 10) AS group#16L]
            +- Range (1, 1000001, step=1, splits=4)


[Screenshot output omitted from public copy; see README.]

### Answers

1. **Purpose of `explain()`:** It prints the query plan so you can inspect how Spark represents and intends to execute a DataFrame query.
2. **Does `explain()` execute the DataFrame and return its records?** No. It prints plan information. An action such as `show()` or `count()` is needed to execute the query and return a result.
3. **Logical Plan vs. Physical Plan:** The logical plan describes what the query means and its transformations. The physical plan chooses concrete operators and execution strategies for producing the result.
4. **What does `Exchange` generally indicate?** A data redistribution boundary, commonly a shuffle. The observed physical plan contains `Exchange hashpartitioning(group#16L, 200)`, which redistributes grouped data between partitions.
5. **Why can `groupBy()` require an Exchange/shuffle?** All rows for a grouping key must meet in the same downstream partition so Spark can finish computing that key's aggregate. A partial aggregate can reduce data before the exchange, but the final grouping still needs compatible partitioning.


## Part 5 - Jobs, Stages, Tasks, and Spark UI (15 Points)


In [8]:
sc = spark.sparkContext
job_group = "lab4_grouped_show"
sc.setJobGroup(job_group, "Lab 4: grouped.show()")
try:
    grouped.show()
finally:
    sc.setLocalProperty("spark.jobGroup.id", None)
    sc.setLocalProperty("spark.job.description", None)
    sc.setLocalProperty("spark.job.interruptOnCancel", None)

tracker = sc.statusTracker()
job_ids = list(tracker.getJobIdsForGroup(job_group))
print("Jobs associated with grouped.show():", len(job_ids), job_ids)

for job_id in job_ids:
    info = tracker.getJobInfo(job_id)
    if info is None:
        print("Job details are no longer available for job", job_id)
        continue
    stage_ids = list(info.stageIds)
    print("Job", job_id, "status:", info.status, "stage IDs:", stage_ids)
    for stage_id in stage_ids:
        stage_info = tracker.getStageInfo(stage_id)
        if stage_info is None:
            print("  Stage", stage_id, "details not available")
        else:
            print("  Stage", stage_id, "tasks:", stage_info.numTasks)

print("Spark UI URL:", sc.uiWebUrl)

Jobs associated with grouped.show(): 4 [16, 15, 4, 3]
Job 16 status: SUCCEEDED stage IDs: [22, 23]
  Stage 22 tasks: 4 (skipped in the UI)
  Stage 23 tasks: 1 (completed)
Job 15 status: SUCCEEDED stage IDs: [21]
  Stage 21 tasks: 4 (completed)
Job 4 status: SUCCEEDED stage IDs: [5, 6]
  Stage 5 tasks: 4 (skipped in the UI)
  Stage 6 tasks: 1 (completed)
Job 3 status: SUCCEEDED stage IDs: [4]
  Stage 4 tasks: 4 (completed)
Spark UI URL: [local Spark UI URL omitted]


## Part 5 Answers

1. **How many jobs were associated with `grouped.show()`?** The tracker reports four jobs in the reused `lab4_grouped_show` group: Jobs 3, 4, 15, and 16. The Spark UI shows two executions: Jobs 3 and 4 at 22:25:25, followed by Jobs 15 and 16 at 22:29:49. Thus, each `grouped.show()` execution produced two jobs.
2. **How many stages did the selected job contain?** Job 4 contains two stages: Stage 5 was skipped, and Stage 6 completed.
3. **How many tasks ran in each stage?** For Job 4, Stage 5's four tasks were skipped and Stage 6 completed one task. Job 3's Stage 4 completed four tasks. In the second execution, Job 16's Stage 22 was skipped (four tasks), Stage 23 completed one task, and Job 15's Stage 21 completed four tasks. The tracker reports stage task counts; the Spark UI distinguishes completed from skipped tasks.
4. **Why can a shuffle cause Spark to create a new stage?** A shuffle redistributes records across partitions so rows with the same key can be processed together. Downstream work depends on upstream shuffle output, so Spark separates the work at that boundary.
5. **Which Spark component executes individual tasks?** Executors execute tasks. The Executors page shows one active executor (the driver) with 16 cores and 122 completed tasks for the application.
6. **Which Spark component coordinates the application?** The driver coordinates the application by planning work, scheduling jobs and stages, and assigning tasks to executors.

**Application → Jobs → Stages → Tasks → Executors**

## Part 6 - Caching Experiment (10 Points)


In [9]:
import time

large_df = spark.range(0, 10000000, numPartitions=8)
processed = large_df.filter(col("id") % 2 == 0)

start = time.time()
first_count = processed.count()
first_time = time.time() - start
print("First execution:", first_time, "seconds; count:", first_count)

processed.cache()
processed.count()  # Materialize the cache before timing the repeated action.

start = time.time()
cached_count = processed.count()
cached_time = time.time() - start
print("Cached execution:", cached_time, "seconds; count:", cached_count)

processed.unpersist()

First execution: 0.06624436378479004 seconds; count: 5000000
Cached execution: 0.05950284004211426 seconds; count: 5000000


DataFrame[id: bigint]

[Screenshot output omitted from public copy; see README.]

### Answers

1. **First execution time:** The first execution took **0.06624436378479004 seconds**.
2. **Cached execution time:** The cached repeat took **0.05950284004211426 seconds**.
3. **Was cached faster?** Yes, the cached repeat was slightly faster in this run. The cache-fill action was timed separately, and timings vary by environment, so one measurement is not conclusive.
4. **Why can caching improve repeated operations?** If a DataFrame is reused, caching can let later actions read already-computed partitions instead of recalculating the full lineage.
5. **Should every DataFrame be cached?** No. Caching consumes executor memory and possibly disk, and cache setup has a cost. Cache data that is reused enough to justify that cost, then unpersist it when finished.

## Part 7 - Efficient vs. Inefficient Spark Operations (10 Points)

### `groupByKey()` vs. `reduceByKey()`

1. **Which is generally preferred for aggregation?** `reduceByKey()` is generally preferred when the goal is to combine values by key.
2. **Why can `groupByKey()` move more data?** It transfers all values for each key to a reducer before aggregation, which can send many records across the network and consume more memory.
3. **How can `reduceByKey()` reduce transfer?** It combines values locally within each map partition before sending partial aggregates across the shuffle, reducing data when keys repeat within partitions.
4. **Which is more scalable for very large datasets?** Usually `reduceByKey()` for associative aggregations such as sum. Use the operation that matches the required result; `groupByKey()` is needed only when the full list of values is genuinely required.

### `collect()` vs. `show(20)`

5. **What does `collect()` do?** It returns every row to the driver as a local collection.
6. **Why can it be dangerous for a very large dataset?** The full result must fit in driver memory and its transfer can overwhelm the driver or network.
7. **Why is `show()` safer for inspection?** `show(20)` displays only a small number of rows, so it avoids transferring the entire dataset to the driver. It still runs a Spark action, but limits the returned sample.


## Part 8 - Spark Optimization Challenge (5 Points)

The inherited application reads several terabytes from CSV, collects the complete input to the driver, and then performs a distributed aggregation. The following changes improve scale and reduce unnecessary work.

| Recommendation | Change | Why it helps |
|---|---|---|
| Remove `df.collect()` | Delete it unless the complete dataset is deliberately small and driver-resident. For inspection, use `limit(...).show()`; for output, write results to distributed storage. | Avoids driver-memory exhaustion and a huge transfer from executors to the driver. |
| Use a typed columnar format | Convert recurring source data to Parquet (or another supported columnar format) with an explicit schema. | Parquet stores types and supports column pruning and predicate pushdown, reducing parsing and I/O when the query can use those features. |
| Define the schema | Supply the expected schema instead of repeatedly inferring CSV types. | Avoids inference work and protects against inconsistent type interpretation. |
| Read only needed columns and filter early | Select `category` and any required fields; apply relevant predicates before aggregation. | Reduces the number of columns and rows scanned, processed, and potentially shuffled. |
| Keep aggregation distributed | Retain `groupBy("category").count()` and write the small result to distributed storage if it is a deliverable. | The aggregation is performed across executors; only the small grouped result needs to be inspected or returned. |
| Tune file and shuffle partitioning from measurements | Inspect input partition sizes, task duration, skew, and shuffle read/write in the UI; adjust file sizing and shuffle partition settings based on evidence. | Avoids too few oversized tasks or too many tiny tasks and helps address skew. |
| Cache only for reuse | Cache only if the same parsed/filtered data is reused in later actions, and unpersist when finished. | Prevents wasted recomputation when useful while avoiding unnecessary memory pressure. |
| Prefer built-in Spark expressions | Express filters, projections, and aggregation with Spark SQL functions instead of Python loops or row-wise UDFs where possible. | Lets Spark optimize the query and avoids per-row Python serialization overhead for logic Spark already supports. |

At least three recommendations are required; the table provides more so you can choose and explain the most relevant ones.


## Part 9 - Reflection (5 Points)


### Q1. Why does Spark use lazy evaluation instead of executing every transformation immediately?

Spark delays transformations until an action requests a result. This lets it build a complete plan before doing the computation. The optimizer can combine compatible operations and push filters or column selection toward the data source when supported. Spark can also avoid work that is not needed for the requested result, saving time and resources.


### Q2. Explain Job -> Stage -> Task in your own words.

An action starts a job when it asks Spark to produce a result. Spark divides that job into stages at dependency boundaries, commonly where a shuffle is needed. Within a stage, Spark normally creates one task for each partition it must process. Executors run those tasks, while the driver plans and coordinates the work.


### Q3. What is a shuffle, and why should Spark developers care about it?

A shuffle redistributes records among partitions so related data can be processed together. Operations such as grouping and joining may need a shuffle when matching records are spread across partitions. Moving data can require network transfer, serialization, memory, and disk reads or writes. Developers should inspect exchanges and shuffle metrics because large or skewed shuffles can become performance bottlenecks.


### Q4. What information can the Spark UI provide that cannot be learned simply by looking at the final result?

The Spark UI shows the jobs, stages, and tasks Spark ran to produce a result. It reports task counts and durations, along with input, output, and shuffle data. Executor information can help show where work ran and whether resources were used unevenly. These details help explain performance problems that are invisible in the final rows alone.


### Q5. What is one optimization technique from this lab that you expect to use in your semester project?

In my semester project, I expect to filter rows and select only needed columns early, inspect the physical plan for exchanges, avoid collecting large datasets to the driver, and cache a DataFrame when I reuse it. I will use the Spark UI and execution plan to check whether these choices reduce unnecessary work, shuffle transfer, or repeated computation.
